In [ ]:
%matplotlib inline

import os
import sys
import pandas as pd
import numpy as np
import datacube
import geopandas as gpd
import rasterio.features
import matplotlib.pyplot as plt
import matplotlib.animation as animation, FuncAnimation
import matplotlib.patheffects as PathEffects
from datacube.utils.masking import make_mask
from matplotlib import colors as mcolours
from IPython.display import Image
from IPython.core.display import Video
from shapely.geometry import shape, box
from skimage.exposure import rescale_intensity
from pathlib import Path

from PIL import ImageSequence
from PIL import Image as pimg

sys.path.insert(1, "../Tools/")
from dea_tools.plotting import xr_animation
from dea_tools.spatial import xr_vectorize, add_geobox
from dea_tools.landcover import lc_animation, lc_colourmap

In [ ]:
dc = datacube.Datacube(app="landcover_geomad_aminations")

In [ ]:
def timeseries_animation(file_name_gmad, file_name_landcover, aspect_ratio=None, crop=False):
    """
    Creates a side-by-side GIF animation combining geomad and land cover animations, ensuring they are synchronized.

    Parameters:
    file_name_gmad (str): File path of the geomad animation GIF.
    file_name_landcover (str): File path of the land cover animation GIF.
    aspect_ratio (float): The desired aspect ratio to crop to (width/height).

    Returns:
    str: File path of the combined animation GIF.
    """
    final_animations_dir = os.path.join(output_dir, 'final_animations')
    if not os.path.exists(final_animations_dir):
        os.makedirs(final_animations_dir)

    directory, filename = os.path.split(file_name_landcover)
    name, ext = os.path.splitext(filename)
    new_filename = f"{name}_gmad{ext}"
    output_filepath = os.path.join(directory, 'final_animations', new_filename)

    gif_lc = pimg.open(file_name_landcover)
    gif_gmad = pimg.open(file_name_gmad)

    if crop is True:
        frames1 = [crop_to_aspect_ratio(frame, aspect_ratio) for frame in ImageSequence.Iterator(gif_gmad)]
        frames2 = [crop_to_aspect_ratio(frame, aspect_ratio) for frame in ImageSequence.Iterator(gif_lc)]

    else:
        frames1 = [frame.copy() for frame in ImageSequence.Iterator(gif_gmad)]
        frames2 = [frame.copy() for frame in ImageSequence.Iterator(gif_lc)]
        

    # Get dimensions of cropped gifs
    frame_width, frame_height = frames1[0].size

    # Set the figure size dynamically based on the size of the gifs
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(frame_width * 2 / dpi, frame_height / dpi))

    # Function to make sure gif animations are synced
    def frame_update(frame):
        ax1.clear()
        ax2.clear()
        ax1.imshow(frames1[frame % len(frames1)])
        ax2.imshow(frames2[frame % len(frames2)])
        ax1.axis('off')
        ax2.axis('off')

    # Adjust layout to minimize white space
    plt.subplots_adjust(wspace=0, hspace=0, left=0, right=1, top=1, bottom=0)

    # Animation object
    animation_object = animation.FuncAnimation(fig, 
                                               frame_update, 
                                               frames=len(frames1),  
                                               interval=interval)  # Adjust interval as needed

    animation_object.save(output_filepath, writer="Pillow")
    plt.close()

    return output_filepath